13. The Nudged Elastic Band (NEB) method is a computational technique used to determine the minimum‑energy path (MEP) between two known states of a system—typically an initial configuration and a final configuration connected by some physical process such as diffusion, adsorption, desorption, or a chemical reaction. Instead of guessing the transition pathway, NEB constructs a series of intermediate “images” of the system and links them together like an elastic band. During optimization, forces perpendicular to the path push the images toward the true MEP, while spring forces along the path maintain a smooth connection between them. The result is a physically meaningful reaction pathway and an accurate estimate of the activation energy barrier, which is essential for understanding kinetics, transition mechanisms, and rate‑limiting steps in materials and surface processes.

An optimized initial geometry is essential because all subsequent calculations—forces, stresses, electronic structure, and transition pathways—depend sensitively on how close the system is to its true equilibrium configuration. If the starting structure contains unrealistic bond lengths, strained angles, or residual forces, the SCF cycle may become unstable, the optimizer may take inefficient or incorrect steps, and the system may relax toward an artificial local minimum rather than the physically meaningful one. Poor initial geometries also distort quantities such as surface dipoles, work functions, reaction barriers, and NEB pathways, since these properties are defined relative to the equilibrium atomic arrangement. By ensuring that the initial and final geometries are fully relaxed with respect to the chosen method, basis set, and pseudopotentials, we eliminate spurious stresses and guarantee that all subsequent simulations begin from a consistent, physically accurate reference state.

In [ ]:
from ase.build import graphene
from ase.visualize import view
from ase.calculators.siesta import Siesta
import os
from ase.units import Ry
import numpy as np
import matplotlib.pyplot as plt

os.mkdir("gr_siesta_kpts")
os.chdir("gr_siesta_kpts")

# Build graphene with total 8 Å vacuum along z
atoms = graphene(a=2.46, vacuum=8.0)
# Optional: center the sheet exactly in the cell along z
atoms.center(axis=2)

kk_array = []
energy_array = []

for kk in range(1, 15, 1): # loop over k-points from 1x1x1 to 8x8x8
 
    atoms.calc = Siesta(label='c2',
                            xc='PBE',
                            pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                            pseudo_qualifier='',
                            symlink_pseudos=True,
                            mesh_cutoff=200 * Ry,
                            energy_shift=0.01 * Ry,
                            basis_set='DZP',
                            spin='non-polarized',
                            kpts=(kk, kk, 1),
                            fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 100},
                )
    energy = atoms.get_total_energy()
    print("k-points:", kk)
    kk_array.append(kk)
    print("Energy:", energy)
    energy_array.append(energy)

# Plot k-point energy convergence

plt.figure(figsize=(10, 6))  # Set the figure size (optional)
plt.plot(kk_array, energy_array, marker='o', linestyle='-', color='b')  # Plot the data
plt.title('Energy vs k-points')  # Add a title to the plot
plt.xlabel('k-points')  # Label for the x-axis
plt.ylabel('Energy')  # Label for the y-axis
plt.grid(True)  # Add a grid (optional)
plt.show()  # Display the plot

os.chdir("../")

view(atoms, viewer='x3d')

Initial geometry of Ni on graphene

In [ ]:
from ase.build import graphene
from ase.visualize import view
from ase.calculators.siesta import Siesta
import os
from ase.optimize.bfgs import BFGS
from ase.units import Ry
from ase import Atom
from ase.io import write

# Build graphene with total 8 Å vacuum along z
atoms = graphene(a=2.46, vacuum=8.0)
# Optional: center the sheet exactly in the cell along z
atoms.center(axis=2)

atoms = atoms.repeat((3,3,1))
atoms.append(Atom('Ni', (2.5, 1.5, 10.0)))

os.mkdir("gr1_siesta_opt")
os.chdir("gr1_siesta_opt")

atoms.calc = Siesta(label='c2',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(3, 3, 1),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 200},
               )

energy = atoms.get_total_energy()

opt = BFGS(atoms, trajectory='atoms.traj')
opt.run(fmax=0.03)

os.chdir("../")

write('gr1_siesta.traj', atoms)

view(atoms, viewer='x3d')

Final geometry

In [ ]:
from ase.build import graphene
from ase.visualize import view
from ase.calculators.siesta import Siesta
import os
from ase.optimize.bfgs import BFGS
from ase.units import Ry
from ase import Atom
from ase.io import write
from ase.io import read

atoms = read('gr1_siesta.traj')
mg_index = [i for i, a in enumerate(atoms) if a.symbol == 'Ni'][0]
dx = 1.5
dy = 2

atoms[mg_index].position += [dx, dy, 0.0]

os.mkdir("gr2_siesta_opt")
os.chdir("gr2_siesta_opt")

atoms.calc = Siesta(label='c2',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(3, 3, 1),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 200},
               )

energy = atoms.get_total_energy()

opt = BFGS(atoms, trajectory='atoms.traj')
opt.run(fmax=0.03)

os.chdir("../")

write('gr2_siesta.traj', atoms)

view(atoms, viewer='x3d')

The NEB calculation proceeds by first reading the initial and final geometries and constructing a series of intermediate images using a linear interpolation algorithm. These images define the discrete points along the reaction pathway. Each image is then assigned its own SIESTA calculator so that forces and energies are evaluated consistently and independently. With the band fully defined, the NEB optimizer adjusts the images by removing the parallel component of the true force and adding spring forces along the path, gradually pushing the band toward the minimum‑energy pathway. Once the optimization converges, the code extracts the energies of all images, plots the NEB energy profile, and identifies the transition state as the image with the highest energy. Finally, the geometry of this transition‑state image is written out for visualization and further analysis, providing a complete picture of the reaction barrier and the atomic configuration at the saddle point.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from ase.io import read, write
from ase.mep import NEB
from ase.optimize import BFGS
from ase.calculators.siesta import Siesta
from ase.units import Ry
from ase.visualize import view


# ------------------------------------------------------------
# 1. Read initial and final geometries
# ------------------------------------------------------------
initial = read('gr1_siesta.traj')   # or initial.xyz / initial.traj
final   = read('gr2_siesta.traj')

os.mkdir("neb_siesta")
os.chdir("neb_siesta")

# ------------------------------------------------------------
# 2. Create NEB images (initial + 3 intermediates + final)
# ------------------------------------------------------------
n_images = 5
images = [initial]

for i in range(n_images - 2):
    images.append(initial.copy())

images.append(final)

neb = NEB(images)
neb.interpolate()   # linear interpolation

# ------------------------------------------------------------
# 3. Attach a *separate* SIESTA calculator to each image
# ------------------------------------------------------------
def make_siesta_calc(label):
    return Siesta(label='c2',
                        xc='PBE',
                        pseudo_path=os.environ["SIESTA_PSEUDO_DIR"],
                        pseudo_qualifier='',
                        symlink_pseudos=True,
                        mesh_cutoff=230 * Ry,
                        energy_shift=0.01 * Ry,
                        basis_set='DZP',
                        spin='non-polarized',
                        kpts=(3, 3, 1),
                        fdf_arguments={'DM.MixingWeight': 0.1, 'MaxSCFIterations': 200},
               )


for i, img in enumerate(images):
    img.calc = make_siesta_calc(f'neb_img_{i}')

# ------------------------------------------------------------
# 4. Optimize the NEB path
# ------------------------------------------------------------
opt = BFGS(neb, logfile='neb.log')
opt.run(fmax=0.05)  # convergence criterion for forces on images

# ------------------------------------------------------------
# 5. Collect energies and compute relative profile
# ------------------------------------------------------------
energies = [img.get_potential_energy() for img in images]
E0 = energies[0]
rel_energies = [E - E0 for E in energies]

# Identify TS (highest energy image)
ts_index = int(np.argmax(energies))
ts_image = images[ts_index]
write('TS.traj', ts_image)

os.chdir("../")

# ------------------------------------------------------------
# 6. Plot NEB energy profile
# ------------------------------------------------------------
plt.figure()
plt.plot(range(n_images), rel_energies, '-o')
plt.xlabel('Image index')
plt.ylabel('Relative energy (eV)')
plt.title('NEB Energy Profile')
plt.grid(True)
plt.tight_layout()
plt.savefig('neb_profile.png', dpi=200)

print("Absolute energies (eV):", energies)
print("TS image index:", ts_index)
print("TS geometry saved as TS.traj")

view(ts_image, viewer='x3d')